# Préparation SFT — UltraMedical-Preference (anglais)

Partie **anglaise** du SFT : questions de [TsinghuaC3I/UltraMedical-Preference](https://huggingface.co/datasets/TsinghuaC3I/UltraMedical-Preference)
(licence MIT), avec leur meilleure réponse (`chosen`) comme réponse attendue.

- **Pourquoi ce jeu** : il contient des **vignettes cliniques** (MedQA : un patient, ses constantes, une décision) et
  des **patients qui décrivent leurs symptômes** (ChatDoctor), proches de l'entrée du triage. MedQuAD, l'autre jeu
  anglais, ne contient que des questions générales sur les maladies (« What are the symptoms of X? »).
- **Réponses vérifiées** : pour les QCM, la lettre donnée doit être celle du corrigé ; pour toutes les sources, la
  note GPT-4 de la réponse doit être élevée.
- **SFT et DPO séparés** : chaque question est attribuée au SFT ou au DPO par son empreinte (moitié / moitié), et le
  `test` d'UltraMedical, revu par des experts, est réservé au DPO et à l'évaluation clinique.
- **Sélection pour le triage** : mêmes critères que le corpus français (`projet_14.selection`), budget de 2 000 paires.
- **Sortie** : `data/v4/sft_ultramedical_{train,validation,test}.jsonl`, puis la version anonymisée dans `data/v4/anonymized/`.

# Import

In [1]:
import re
from pathlib import Path

import pandas as pd
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer

from projet_14.conversation import SYSTEM_PROMPT_EN
from projet_14.mediqal import case_assignment, case_id
from projet_14.rgpd import run
from projet_14.selection import case_table, select_cases

OUT = Path("../data/v4")
STEM = "sft_ultramedical"
MODEL_NAME = "Qwen/Qwen3-1.7B-Base"  # pour compter les tokens avec le tokenizer du modèle entraîné

# Sources gardées : vignettes cliniques (QCM d'examen) et patients qui décrivent leurs symptômes
SOURCES = ["MedQA", "MedQA-Evol", "MedMCQA", "ChatDoctor"]
MCQ_SOURCES = ["MedQA", "MedQA-Evol", "MedMCQA"]  # bonne réponse (lettre) vérifiable dans la réponse
MIN_CHOSEN_SCORE = 4.5  # note GPT-4 minimale de la réponse (de 1 à 5)

# Longueurs maximales en tokens : MAX_LENGTH est celui du SFT (SFTConfig.max_length)
MAX_PROMPT_TOKENS = 512
MAX_LENGTH = 1_536
TEMPLATE_TOKENS = 40  # consigne système et balises du gabarit de conversation

# Sélection pour le triage
BUDGET = 2_000           # paires retenues, tous splits confondus
MAX_SOURCE_SHARE = 0.4   # part maximale du budget par source

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chargement

Parties `train` et `validation` seulement (environ 1 Go, mis en cache). Le `test` (777 paires revues par des experts,
Medical RewardBench) est réservé au DPO et à l'évaluation clinique : il n'est pas chargé ici.

In [2]:
ds = load_dataset("TsinghuaC3I/UltraMedical-Preference")


def flatten(d: pd.DataFrame) -> pd.DataFrame:
    # Une ligne par paire : question, meilleure réponse et ses métadonnées
    meta = pd.json_normalize(d["metadata"].tolist())
    return pd.DataFrame({
        "prompt_id": d["prompt_id"],
        "source": d["prompt_id"].str.split(",").str[0],  # "MedQA,1234" -> "MedQA"
        "prompt": d["prompt"],
        "answer": d["chosen"].map(lambda messages: messages[-1]["content"]),
        "chosen_model": meta["chosen.model"],
        "chosen_score": meta["chosen.score"],
        "golden_answer": meta["golden_answer"],
    })


df = pd.concat([flatten(ds[name].to_pandas()) for name in ("train", "validation")], ignore_index=True)

# Questions du test présentes aussi en train / validation (même texte, autre identifiant) : écartées
RESERVED = set(ds["test"]["prompt"])
reserved = df["prompt"].isin(RESERVED)
print(f"questions du test présentes en train / validation : {reserved.sum()} paires écartées")
df = df[~reserved]
df.shape

questions du test présentes en train / validation : 44 paires écartées


(111541, 7)

# Analyse

In [3]:
# Une même question apparaît dans plusieurs paires (easy, hard, length) : une seule ligne par question sera gardée
print(f"{len(df)} paires, {df['prompt'].nunique()} questions distinctes\n")
df["source"].value_counts()

111541 paires, 77494 questions distinctes



source
WikiInstruct             28050
MedMCQA                  17892
TextBookQA               16345
ChatDoctor               11544
MedQA                    11028
MedQA-Evol                9446
MedQuad                   5948
Medical-Instruct-120k     5034
PubMedQA                  3359
MedInstruct-52k           2895
Name: count, dtype: int64

# Nettoyage et qualité des réponses

- Espaces en trop retirés ; réponses vides écartées.
- **QCM** : la lettre donnée par la réponse (« So, the answer is C. ») doit être celle du corrigé.
- **Toutes les sources** : note GPT-4 de la réponse ≥ `MIN_CHOSEN_SCORE`.
- **Une ligne par question** : la réponse la mieux notée.

In [4]:
df["prompt"] = df["prompt"].str.strip()
df["answer"] = df["answer"].str.strip()

ANSWER = re.compile(r"answer is:?\s*\(?\**([A-J])\b", re.I)


def answer_letter(text: str) -> str | None:
    # Lettre donnée comme réponse finale, ou None si absente
    found = ANSWER.findall(text)
    return found[-1].upper() if found else None


is_mcq = df["source"].isin(MCQ_SOURCES)
correct = df["answer"].map(answer_letter) == df["golden_answer"]
good = (~is_mcq | correct) & (df["chosen_score"] >= MIN_CHOSEN_SCORE) & (df["answer"] != "")
print(f"réponses vérifiées et bien notées : {good.sum()} sur {len(df)}")
df = df[good].assign(pair_check=lambda d: d["source"].isin(MCQ_SOURCES).map({True: "golden_answer", False: "gpt4_score"}))
df = df.sort_values("chosen_score", ascending=False, kind="stable").drop_duplicates(subset="prompt")
df.shape

réponses vérifiées et bien notées : 73924 sur 111541


(55964, 8)

# Filtres

- **Sources** : vignettes cliniques et patients (`SOURCES`). Écartées : articles et synthèses (WikiInstruct,
  PubMedQA), manuels (TextBookQA), questions générales (MedQuad) et consignes génériques (MedInstruct, Medical-Instruct).
- **Longueur** : une réponse coupée par `max_length` perdrait sa fin, et avec elle `<|endoftext|>` : le modèle
  apprendrait à ne pas s'arrêter. On ne garde que les exemples qui tiennent entiers.

In [5]:
df = df[df["source"].isin(SOURCES)]
print(f"sources gardées : {len(df)} questions")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def n_tokens(texts: pd.Series) -> list[int]:
    return [len(ids) for ids in tokenizer(texts.tolist(), add_special_tokens=False)["input_ids"]]


df = df.assign(n_prompt=n_tokens(df["prompt"]), n_answer=n_tokens(df["answer"]))
fits = (df["n_prompt"] <= MAX_PROMPT_TOKENS) & (df["n_prompt"] + df["n_answer"] + TEMPLATE_TOKENS <= MAX_LENGTH)
print(f"trop longues : {(~fits).sum()}")
df = df[fits]
df.groupby("source")[["n_prompt", "n_answer"]].median().astype(int)

sources gardées : 24269 questions


trop longues : 50


,n_prompt,n_answer
source,,
ChatDoctor,102,546
MedMCQA,60,287
MedQA,196,318
MedQA-Evol,131,339


# Répartition SFT / DPO et train / validation / test

Comme pour MediQAl (`projet_14.mediqal.case_assignment`), l'empreinte de la question donne deux tirages
indépendants : l'**usage** (SFT ou DPO, moitié / moitié) et le **split** (70 / 10 / 20). La moitié DPO n'est jamais
utilisée ici : le futur jeu DPO anglais n'aura aucune question commune avec le SFT.

In [6]:
df["case_id"] = df["prompt"].map(case_id)
assignment = df["case_id"].map(case_assignment)
df["usage"], df["split"] = assignment.str[0], assignment.str[1]
print(df["usage"].value_counts().to_dict())
df = df[df["usage"] == "sft"]
pd.crosstab(df["source"], df["split"], margins=True)

{'dpo': 12181, 'sft': 12038}


split,test,train,validation,All
source,,,,
ChatDoctor,417,1487,182,2086
MedMCQA,890,3123,448,4461
MedQA,529,1977,275,2781
MedQA-Evol,547,1886,277,2710
All,2383,8473,1182,12038


# Conversations

Un échange par question : consigne système (anglais), question, réponse. Même format `messages` que le corpus
français, avec les mêmes métadonnées principales (`ids`, `case_id`, `n_questions`).

In [7]:
examples = pd.DataFrame({
    "ids": df["prompt_id"].map(lambda i: [i]),
    "source": df["source"],
    "case_id": df["case_id"],
    "split": df["split"],
    "n_questions": 1,
    "pair_check": df["pair_check"],
    "golden_answers": df["golden_answer"].map(lambda g: [g] if isinstance(g, str) and g else []),
    "chosen_model": df["chosen_model"],
    "chosen_score": df["chosen_score"],
    "lang": "en",
    "messages": [
        [{"role": "system", "content": SYSTEM_PROMPT_EN}, {"role": "user", "content": p}, {"role": "assistant", "content": a}]
        for p, a in zip(df["prompt"], df["answer"])
    ],
})
examples.shape

(12038, 11)

# Sélection pour le triage

Mêmes critères que le corpus français (`projet_14.selection`), sur la question et la réponse :

1. **Patient décrit** (obligatoire) : vignette (« A 45-year-old man… ») ou patient qui parle de lui ou d'un proche.
2. **Score de pertinence** (0 à 2) : **constantes** dans la question (blood pressure, pulse, SpO2, temperature…) +
   vocabulaire d'**urgence** ou de conduite à tenir (emergency, next step, admission, ICU, shock…).
3. **Budget** de `BUDGET` paires, réparti entre les splits (70 / 10 / 20) : par score décroissant (score ≥ 1), puis
   dans un ordre pseudo-aléatoire fixe tiré de l'identifiant.
4. **Plafond par source** : `MAX_SOURCE_SHARE` du budget au plus, pour mêler vignettes d'examen et patients.

In [8]:
cases = case_table(examples, lang="en", group="source")
# Questions disponibles selon le patient décrit et le score
print(pd.crosstab(cases["patient"].map({True: "patient décrit", False: "sans patient"}), cases["score"], margins=True), "\n")

selected = select_cases(cases, BUDGET, MAX_SOURCE_SHARE, group="source")
examples = examples[examples["case_id"].isin(selected["case_id"])]
print(f"retenues : {len(selected)} paires (budget {BUDGET})")
selected.groupby("split").agg(paires=("case_id", "size"), constantes=("constantes", "mean"), urgence=("urgence", "mean")).round(2)

score              0     1     2    All
patient                                
patient décrit  4128  2539   979   7646
sans patient    3666   658    68   4392
All             7794  3197  1047  12038 

retenues : 2000 paires (budget 2000)


,paires,constantes,urgence
split,,,
test,400,0.55,0.90
train,1400,0.56,0.94
validation,200,0.57,0.93


In [9]:
# Sources retenues : aucune ne dépasse MAX_SOURCE_SHARE du budget
pd.crosstab(selected["source"], selected["split"], margins=True)

split,test,train,validation,All
source,,,,
ChatDoctor,94,325,48,467
MedMCQA,49,170,21,240
MedQA,160,560,80,800
MedQA-Evol,97,345,51,493
All,400,1400,200,2000


In [10]:
# Un exemple, tel que le verra le modèle
for message in examples.iloc[0]["messages"]:
    print(f"--- {message['role']}\n{message['content'][:800]}\n")

--- system
You are an expert medical assistant. Answer accurately and clearly.

--- user
A 22-year-old medical student presents to a community health center due to an episode of loss of consciousness 3 days ago. She also has a history of multiple episodes of dizziness in the last year. These episodes almost always occur when she is observing surgery in the operating room. She describes her dizziness as a feeling of lightheadedness, warmth, excessive sweating, and palpitations. She feels that she will fall down if she stood longer and usually sits on the floor or leaves the room until the feeling subsides. Three days ago, she collapsed while observing an open cholecystectomy but regained consciousness after a few seconds. Once she regained consciousness, she was pale and sweating excessively. Her medical history is significant for migraines, but she is not on prophylactic thera

--- assistant
The patient is experiencing episodes of dizziness and syncope predominantly in situations that 

# Export JSONL

In [12]:
OUT.mkdir(parents=True, exist_ok=True)
METADATA = ["ids", "source", "case_id", "n_questions", "pair_check", "golden_answers", "chosen_model", "chosen_score", "lang"]
for split in ("train", "validation", "test"):
    part = examples[examples["split"] == split]
    Dataset.from_pandas(part[METADATA + ["messages"]], preserve_index=False).to_json(
        OUT / f"{STEM}_{split}.jsonl", force_ascii=False)
    print(split, len(part), "lignes")

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 17.35ba/s]

Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 17.16ba/s]

train 1400 lignes


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 91.81ba/s]

validation 200 lignes


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 53.27ba/s]

test 400 lignes


# Anonymisation des données

Seule la source **ChatDoctor** contient de vrais échanges entre patients et médecins (« My name is Roger… ») : elle
est anonymisée en entier (question et réponse, qui reprend souvent le prénom). Les vignettes d'examen sont fictives.

In [13]:
reports = {
    split: run(OUT / f"{STEM}_{split}.jsonl", lang="en", output_dir=OUT / "anonymized", sources=["ChatDoctor"])
    for split in ("train", "validation", "test")
}
pd.DataFrame(reports).T[["records", "records_scanned", "records_modified", "persons_replaced"]]

,records,records_scanned,records_modified,persons_replaced
train,1400,325,138,279
validation,200,48,24,49
test,400,94,38,78


# Contrôle final

In [14]:
final = load_dataset("json", data_files={s: str(OUT / "anonymized" / f"{STEM}_{s}.anonymized.jsonl") for s in ("train", "validation", "test")})
print(final)

# Aucune question partagée entre les splits ; aucune de la moitié DPO ni du test d'UltraMedical
ids = {s: set(d["case_id"]) for s, d in final.items()}
assert not ids["train"] & (ids["validation"] | ids["test"]), "question partagée entre train et évaluation"
assert not ids["validation"] & ids["test"], "question partagée entre validation et test"
assert all(case_assignment(cid)[0] == "sft" for s in ids.values() for cid in s), "question réservée au DPO"
reserved_ids = {case_id(p.strip()) for p in RESERVED}
assert not reserved_ids & set().union(*ids.values()), "question du test d'UltraMedical"
print("OK : aucune question partagée entre les splits, avec le DPO ou avec le test d'UltraMedical")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 1400 examples [00:00, 45302.20 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating validation split: 200 examples [00:00, 22895.30 examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 400 examples [00:00, 12369.11 examples/s]

DatasetDict({
    train: Dataset({
        features: ['ids', 'source', 'case_id', 'n_questions', 'pair_check', 'golden_answers', 'chosen_model', 'chosen_score', 'lang', 'messages', 'persons_replaced'],
        num_rows: 1400
    })
    validation: Dataset({
        features: ['ids', 'source', 'case_id', 'n_questions', 'pair_check', 'golden_answers', 'chosen_model', 'chosen_score', 'lang', 'messages', 'persons_replaced'],
        num_rows: 200
    })
    test: Dataset({
        features: ['ids', 'source', 'case_id', 'n_questions', 'pair_check', 'golden_answers', 'chosen_model', 'chosen_score', 'lang', 'messages', 'persons_replaced'],
        num_rows: 400
    })
})
OK : aucune question partagée entre les splits, avec le DPO ou avec le test d'UltraMedical


# Limites

- **Réponses générées par des LLM** (GPT-4, Llama-3…), jugées par GPT-4. Pour les QCM, la bonne réponse est
  vérifiée par le corrigé ; le raisonnement qui y mène ne l'est pas.
- **Réponses longues** (plusieurs centaines de tokens) alors que le corpus français a des réponses courtes : le
  modèle apprend deux styles, selon la langue et le type de question.
- **Contexte américain** (USMLE) : unités, médicaments et organisation des soins diffèrent parfois des protocoles
  français.